# Step 2 — Weather processing: LCD to an hourly, UTC-aligned table

Builds the hourly, UTC-aligned weather table used by all experiments (paper Sect. 3.2 and Table 3).

- **Input:** `data/raw/noaa_lcd/<year>/*.csv` (from step 1)
- **Intermediate:** `data/interim/weather_parquet/weather_<year>.parquet`
- **Output:** `data/processed/weather_hourly_utc.parquet` (one row per station-hour)

Alignment issues corrected in this notebook (see Table 3 of the paper):

| # | Issue in a common BTS–NOAA join | Correction |
|---|---|---|
| 1 | LST→UTC conversion by state (ELP, PNS, ECP, VPS off by 1 h) | Per-airport IANA time zone |
| 2 | Precipitation summed over `FM-15` and `FM-16` (≈2.2× inflation) | Precipitation from `FM-15` only |
| 3 | Substring matching of weather codes (mist/haze counted as fog, vicinity thunder as on-site) | Token-level METAR parsing |
| 4 | Obscured-sky (`VV`) ceilings ignored | `VV` treated as a ceiling |
| 5 | Empty sky field treated as clear sky | Treated as missing |
| 6 | Physically impossible values (e.g., wind 2,237 mph) | Range checks |
| 7 | Unbounded forward/backward filling | Forward fill ≤ 3 h, no backward fill |
| 8 | Flight category from ceiling only | FAA definition (ceiling **and** visibility) |

### Imports
Requires `duckdb`, `airportsdata` and `tzdata` (see `requirements.txt`).

In [1]:
import re
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import duckdb
import numpy as np
import pandas as pd
import airportsdata

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'README.md').exists())  # project root

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)


### Paths and settings

In [2]:
WEATHER_CSV_ROOT = ROOT / 'data' / 'raw' / 'noaa_lcd'           # output of step 1
WEATHER_PARQUET_ROOT = ROOT / 'data' / 'interim' / 'weather_parquet'
OUT_DIR = ROOT / 'data' / 'processed'
WEATHER_CLEAN_PATH = OUT_DIR / 'weather_hourly_utc.parquet'

YEARS = [2018, 2019, 2020, 2021, 2022]
TOP10_STATES = ['CA', 'TX', 'FL', 'IL', 'GA', 'NY', 'CO', 'NC', 'VA', 'WA']
for d in (WEATHER_PARQUET_ROOT, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)


### Convert yearly CSV files to Parquet
DuckDB reads all station files of a year, keeps the ten study states and hourly report types (`FM-15`, `FM-16`), and writes one Parquet file per year. Existing files are skipped.

In [3]:
# DuckDB is used only for fast CSV -> Parquet conversion
con = duckdb.connect(database=':memory:')
con.execute("SET threads TO 8;")

def csv_to_parquet_year(year: int, overwrite: bool = False):
    out_path = WEATHER_PARQUET_ROOT / f'weather_{year}.parquet'
    if out_path.exists() and not overwrite:
        print(f'{year}: {out_path.name} exists, skipped')
        return

    csv_glob = (WEATHER_CSV_ROOT / str(year) / '*.csv').as_posix()
    states_list = ','.join(f"'{s}'" for s in TOP10_STATES)

    query = f"""
    COPY (
        SELECT
            regexp_extract(filename, '([A-Z]{{2}})_[A-Z0-9]{{3,4}}_\\d{{4}}\\.csv$', 1) AS state,
            regexp_extract(filename, '[A-Z]{{2}}_([A-Z0-9]{{3,4}})_\\d{{4}}\\.csv$', 1) AS iata,
            {year} AS year,
            STATION, DATE, NAME, LATITUDE, LONGITUDE, ELEVATION,
            REPORT_TYPE, SOURCE,
            HourlyDewPointTemperature, HourlyDryBulbTemperature,
            HourlyPrecipitation, HourlyPresentWeatherType,
            HourlyRelativeHumidity, HourlySkyConditions,
            HourlySeaLevelPressure, HourlyStationPressure,
            HourlyPressureChange, HourlyVisibility,
            HourlyWetBulbTemperature,
            HourlyWindDirection, HourlyWindGustSpeed, HourlyWindSpeed
        FROM read_csv(
            '{csv_glob}',
            all_varchar = true,
            filename = true,
            union_by_name = true,
            ignore_errors = true
        )
        WHERE regexp_extract(filename, '([A-Z]{{2}})_[A-Z0-9]{{3,4}}_\\d{{4}}\\.csv$', 1)
              IN ({states_list})
          AND TRIM(REPORT_TYPE) IN ('FM-15', 'FM-16')
    )
    TO '{out_path.as_posix()}' (FORMAT PARQUET, COMPRESSION SNAPPY);
    """
    print(f'→ {year}: converting...')
    con.execute(query)
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_path.as_posix()}')").fetchone()[0]
    print(f'{year}: {n:,} rows, {out_path.stat().st_size/1e6:.1f} MB')

for y in YEARS:
    csv_to_parquet_year(y, overwrite=False)


con.close()

→ 2018: converting...
2018: 1,633,076 rows, 25.6 MB
→ 2019: converting...
2019: 1,628,892 rows, 25.4 MB
→ 2020: converting...
2020: 1,623,394 rows, 25.2 MB
→ 2021: converting...
2021: 1,620,747 rows, 25.0 MB
→ 2022: converting...
2022: 1,595,494 rows, 24.3 MB


### Load yearly weather files
Reads only the required columns, drops the invalid airport code `USA`, and strips `REPORT_TYPE` so that `FM-15` and `FM-16` can be distinguished.

In [4]:
cols = ['iata', 'state', 'DATE', 'REPORT_TYPE',
        'HourlyDryBulbTemperature', 'HourlyDewPointTemperature', 'HourlyWetBulbTemperature',
        'HourlyRelativeHumidity', 'HourlyPrecipitation', 'HourlyPresentWeatherType',
        'HourlySkyConditions', 'HourlySeaLevelPressure', 'HourlyStationPressure',
        'HourlyVisibility', 'HourlyWindDirection', 'HourlyWindGustSpeed', 'HourlyWindSpeed']
parts = []
for y in YEARS:
    p = WEATHER_PARQUET_ROOT / f'weather_{y}.parquet'
    if p.exists():
        parts.append(pd.read_parquet(p, columns=cols))
        print(f'  {y}: {len(parts[-1]):,} rows')
df_wx = pd.concat(parts, ignore_index=True)
del parts

df_wx = df_wx[df_wx['iata'] != 'USA'].copy()
df_wx['REPORT_TYPE'] = df_wx['REPORT_TYPE'].astype(str).str.strip()
print(f'Total: {len(df_wx):,} reports, {df_wx["iata"].nunique()} stations')
print(df_wx['REPORT_TYPE'].value_counts())

  2018: 1,633,076 rows
  2019: 1,628,892 rows
  2020: 1,623,394 rows
  2021: 1,620,747 rows
  2022: 1,595,494 rows
Total: 8,101,603 reports, 140 stations
REPORT_TYPE
FM-15    6692538
FM-16    1409065
Name: count, dtype: int64


### Parse timestamps
Timestamps are converted to UTC before rounding to the hour (next cells).

In [5]:
df_wx['DATE'] = pd.to_datetime(df_wx['DATE'], errors='coerce')
df_wx = df_wx.dropna(subset=['DATE'])
print(f'Period (LST): {df_wx["DATE"].min()} → {df_wx["DATE"].max()}')

Period (LST): 2018-01-01 00:00:00 → 2022-12-31 23:59:00


### Numeric conversion, range checks, and precipitation
- Vectorised conversion; trace precipitation (`T`) is set to 0.005 in; suffixes (`s`, `*`, `V`) are removed.
- Values outside physical ranges are set to missing; visibility is capped at 10 mi.
- **Precipitation is taken from routine `FM-15` reports only.** Special `FM-16` reports carry the amount accumulated since the last routine report, so summing both inflates hourly totals.

In [6]:
def to_num(s, trace_value=None):
    s = s.astype('string').str.strip().str.upper()
    if trace_value is not None:
        s = s.mask(s.eq('T').fillna(False), str(trace_value))   # 'T' = trace precipitation
    s = s.str.replace(r'[S\*V]', '', regex=True)                # suspect / variable flags
    return pd.to_numeric(s, errors='coerce')

num_map = {
    'HourlyDryBulbTemperature': 'temp_f', 'HourlyDewPointTemperature': 'dewpt_f',
    'HourlyWetBulbTemperature': 'wetbulb_f', 'HourlyRelativeHumidity': 'humidity',
    'HourlyPrecipitation': 'precip_in', 'HourlySeaLevelPressure': 'slp',
    'HourlyStationPressure': 'pressure', 'HourlyVisibility': 'visibility_mi',
    'HourlyWindDirection': 'wind_dir', 'HourlyWindGustSpeed': 'gust_mph',
    'HourlyWindSpeed': 'wind_mph',
}
for src, dst in num_map.items():
    df_wx[dst] = to_num(df_wx[src], 0.005 if dst == 'precip_in' else None).astype('float32')

# Physical range checks
RANGES = {
    'temp_f': (-60, 130), 'dewpt_f': (-70, 95), 'wetbulb_f': (-60, 100),
    'humidity': (0, 100), 'precip_in': (0, 5), 'slp': (27.5, 31.5),
    'pressure': (19.5, 31.5), 'wind_dir': (0, 360),
    'wind_mph': (0, 100), 'gust_mph': (0, 150),
}
df_wx['visibility_mi'] = df_wx['visibility_mi'].clip(lower=0, upper=10)
for c, (lo, hi) in RANGES.items():
    bad = (df_wx[c] < lo) | (df_wx[c] > hi)
    if bad.sum():
        print(f'  {c}: {bad.sum():,} values outside [{lo}, {hi}] -> NaN')
    df_wx.loc[bad, c] = np.nan

# Wind direction -> sin/cos; calm wind has no direction
df_wx.loc[df_wx['wind_mph'] == 0, 'wind_dir'] = np.nan
df_wx['wind_dir_sin'] = np.sin(np.deg2rad(df_wx['wind_dir'])).astype('float32')
df_wx['wind_dir_cos'] = np.cos(np.deg2rad(df_wx['wind_dir'])).astype('float32')

# Precipitation from routine FM-15 reports only
df_wx.loc[df_wx['REPORT_TYPE'] != 'FM-15', 'precip_in'] = np.nan
df_wx[list(num_map.values())].describe().T[['count', 'mean', 'min', 'max']].round(2)

  dewpt_f: 1 values outside [-70, 95] -> NaN
  precip_in: 1 values outside [0, 5] -> NaN
  wind_mph: 29 values outside [0, 100] -> NaN


,count,mean,min,max
temp_f,8084527.0,60.08,-33.00,128.00
dewpt_f,8077436.0,48.55,-40.00,95.00
wetbulb_f,7844775.0,54.22,-30.00,99.00
humidity,8076965.0,70.61,1.00,100.00
precip_in,5554700.0,0.00,0.00,4.78
slp,5917134.0,30.03,28.74,31.18
pressure,7863398.0,28.95,19.92,30.82
visibility_mi,8080049.0,8.75,0.00,10.00
wind_dir,6484520.0,189.23,0.00,360.00
gust_mph,962936.0,25.71,11.00,135.00


### Parse present-weather codes
LCD present-weather strings (e.g., `"-RA:02 BR:1 |RA |RA"`) are split into METAR tokens (intensity, descriptor, phenomenon) instead of matching substrings:
- `has_fog` = `FG` only; mist (`BR`) is a separate flag `has_mist`; haze (`HZ`) is ignored;
- `has_thunder` = thunderstorm at the station; `has_thunder_vc` = thunderstorm in the vicinity (`VCTS`).

Each distinct string is parsed once and mapped back to all rows.

In [7]:
WX_RE = re.compile(
    r'^(?P<int>[-+]|VC)?(?P<desc>MI|PR|BC|DR|BL|SH|TS|FZ)?'
    r'(?P<ph>(?:DZ|RA|SN|SG|IC|PL|GR|GS|UP|BR|FG|FU|VA|DU|SA|HZ|PY|PO|SQ|FC|SS|DS)*)$')
FLAG_COLS = ['has_rain', 'has_snow', 'has_fog', 'has_mist', 'has_thunder',
             'has_thunder_vc', 'has_freezing', 'is_heavy']

def parse_wx(s):
    out = dict.fromkeys(FLAG_COLS, 0)
    for tok in re.split(r'[\s|]+', str(s)):
        tok = re.sub(r':\d+$', '', tok.strip()).upper()   # '-RA:02' -> '-RA'
        if not tok or tok.isdigit():
            continue
        m = WX_RE.match(tok)
        if not m:
            continue
        inten, desc, ph = m.group('int') or '', m.group('desc') or '', m.group('ph') or ''
        phs = {ph[i:i + 2] for i in range(0, len(ph), 2)}
        if desc == 'TS':
            out['has_thunder_vc' if inten == 'VC' else 'has_thunder'] = 1
        if inten == 'VC':                 # vicinity phenomena are not on-site
            continue
        if phs & {'RA', 'DZ'}:            out['has_rain'] = 1
        if phs & {'SN', 'SG', 'PL', 'IC', 'GR', 'GS'}: out['has_snow'] = 1
        if 'FG' in phs:                   out['has_fog'] = 1
        if 'BR' in phs:                   out['has_mist'] = 1
        if desc == 'FZ':                  out['has_freezing'] = 1
        if inten == '+':                  out['is_heavy'] = 1
    return out

# Examples
for s in ['-RA:02 BR:1 |RA |RA', 'BR:1 ||', 'HZ:7 |HZ |', 'VCTS |', '+TSRA:05 |TSRA |']:
    print(f'{s:22s} ->', {k: v for k, v in parse_wx(s).items() if v})

uniq = df_wx['HourlyPresentWeatherType'].dropna().unique()
flag_tbl = pd.DataFrame.from_dict({s: parse_wx(s) for s in uniq}, orient='index', dtype='int8')
flags = flag_tbl.reindex(df_wx['HourlyPresentWeatherType'].values).fillna(0).astype('int8')
flags.index = df_wx.index
df_wx = pd.concat([df_wx, flags], axis=1)
print(f'Parsed {len(uniq):,} distinct strings')

-RA:02 BR:1 |RA |RA    -> {'has_rain': 1, 'has_mist': 1}
BR:1 ||                -> {'has_mist': 1}
HZ:7 |HZ |             -> {}
VCTS |                 -> {'has_thunder_vc': 1}
+TSRA:05 |TSRA |       -> {'has_rain': 1, 'has_thunder': 1, 'is_heavy': 1}
Parsed 1,948 distinct strings


### Parse sky condition (ceiling)
The cover code is the token before `:` so that vertical visibility (`VV`, obscured sky) is recognised as a ceiling. Empty sky fields are treated as missing rather than clear. Ceilings are capped at 25,000 ft.

In [8]:
COVER = {'CLR': 0, 'SKC': 0, 'FEW': 1, 'SCT': 2, 'BKN': 3, 'OVC': 4, 'VV': 4}

def parse_sky(s):
    if not isinstance(s, str) or not s.strip():
        return (np.nan, np.nan)                       # missing
    toks = s.split()
    cover, ceil, found = 0, 25000.0, False
    for i, t in enumerate(toks):
        code = t.split(':')[0].upper()                # 'VV:09' -> 'VV'
        if code not in COVER:
            continue
        found = True
        cover = max(cover, COVER[code])
        if code in ('BKN', 'OVC', 'VV') and i + 1 < len(toks):
            try:
                ceil = min(ceil, float(toks[i + 1]) * 100)   # '2' -> 200 ft
            except ValueError:
                pass
    return (cover, ceil) if found else (np.nan, np.nan)

print('OVC:08 2   ->', parse_sky('OVC:08 2'))
print('VV:09 2    ->', parse_sky('VV:09 2'))
print('CLR:00     ->', parse_sky('CLR:00'))

uniq = df_wx['HourlySkyConditions'].dropna().unique()
sky_tbl = pd.DataFrame.from_dict({s: parse_sky(s) for s in uniq}, orient='index',
                                 columns=['cloud_max', 'ceiling_ft'])
sky = sky_tbl.reindex(df_wx['HourlySkyConditions'].values)
df_wx['cloud_max'] = sky['cloud_max'].values.astype('float32')
df_wx['ceiling_ft'] = sky['ceiling_ft'].values.astype('float32')

OVC:08 2   -> (4, 200.0)
VV:09 2    -> (4, 200.0)
CLR:00     -> (0, 25000.0)


### Convert local standard time to UTC per airport
LCD timestamps are in local standard time (no daylight saving). The offset of each airport is taken from its IANA time zone (`airportsdata`) in mid-January. Airports whose offset differs from a state-level mapping are printed.

In [9]:
ap = airportsdata.load('IATA')
lst_offset = {}
for code in df_wx['iata'].unique():
    tz = ap.get(code, {}).get('tz')
    if tz is None:
        print(f'  warning: no time zone for {code}')
        continue
    lst_offset[code] = -ZoneInfo(tz).utcoffset(datetime(2021, 1, 15)).total_seconds() / 3600

# Compare with a state-level offset mapping
OLD_STATE_OFFSET = {'CA': 8, 'WA': 8, 'CO': 7, 'TX': 6, 'IL': 6,
                    'FL': 5, 'GA': 5, 'NY': 5, 'NC': 5, 'VA': 5}
state_of = df_wx.groupby('iata')['state'].first()
wrong = {c: (OLD_STATE_OFFSET[state_of[c]], lst_offset[c]) for c in lst_offset
         if OLD_STATE_OFFSET.get(state_of[c]) != lst_offset[c]}
print('Airports misaligned by a state-level offset {iata: (state, airport)}:', wrong)

df_wx = df_wx[df_wx['iata'].isin(lst_offset)].copy()
df_wx['obs_time_utc'] = df_wx['DATE'] + pd.to_timedelta(df_wx['iata'].map(lst_offset), unit='h')
df_wx['obs_hour_utc'] = df_wx['obs_time_utc'].dt.floor('h')

Airports misaligned by a state-level offset {iata: (state, airport)}: {'ECP': (5, 6.0), 'PNS': (5, 6.0), 'VPS': (5, 6.0), 'ELP': (6, 7.0)}


### Hourly aggregation (UTC)
Reports are aggregated by `obs_hour_utc`; `n_reports` counts reports per hour. An hour with no `FM-15` report has missing (not zero) precipitation.

**Convention used downstream:** bucket `b` contains reports in `[b, b+1h)` and is fully observed only at `b+1h`.

In [10]:
agg_rules = {
    'state': 'first',
    'temp_f': 'mean', 'dewpt_f': 'mean', 'wetbulb_f': 'mean', 'humidity': 'mean',
    'slp': 'mean', 'pressure': 'mean',
    'visibility_mi': 'min', 'ceiling_ft': 'min', 'cloud_max': 'max',
    'wind_mph': 'max', 'gust_mph': 'max',
    'wind_dir_sin': 'mean', 'wind_dir_cos': 'mean',
    'precip_in': 'sum',
    **{f: 'max' for f in FLAG_COLS},
}
grp = df_wx.groupby(['iata', 'obs_hour_utc'], sort=True)
df_hourly = grp.agg(agg_rules).reset_index()
df_hourly['n_reports'] = grp.size().values
df_hourly.loc[grp['precip_in'].count().values == 0, 'precip_in'] = np.nan
del grp
print(f'Hourly table: {len(df_hourly):,} rows (station x hour)')

Hourly table: 6,122,624 rows (station x hour)


### Complete hourly grid, bounded filling, and 3-hour summaries
1. A complete hourly grid per station (`wx_missing = 1` for hours without reports).
2. Forward fill limited to 3 hours; no backward filling (which would copy future values into the past).
3. Backward-looking 3-hour summaries (current hour and the two previous hours): `thunder_3h`, `vis_min_3h`, `ceil_min_3h`, `precip_3h`, `wind_max_3h`, `slp_tend_3h`.

In [11]:
cont_cols = ['temp_f', 'dewpt_f', 'wetbulb_f', 'humidity', 'slp', 'pressure',
             'visibility_mi', 'ceiling_ft', 'cloud_max', 'wind_mph', 'gust_mph',
             'wind_dir_sin', 'wind_dir_cos']
FFILL_LIMIT_H = 3

grids = []
for code, g in df_hourly.groupby('iata', sort=False):
    full = pd.date_range(g['obs_hour_utc'].min(), g['obs_hour_utc'].max(), freq='h')
    g = g.set_index('obs_hour_utc').reindex(full)
    g.index.name = 'obs_hour_utc'
    g['iata'] = code
    g['state'] = g['state'].ffill().bfill()             # constant per station
    g['wx_missing'] = g['n_reports'].isna().astype('int8')
    g['n_reports'] = g['n_reports'].fillna(0)
    g['has_gust'] = g['gust_mph'].notna().astype('int8')
    g[cont_cols] = g[cont_cols].ffill(limit=FFILL_LIMIT_H)
    g[FLAG_COLS] = g[FLAG_COLS].ffill(limit=FFILL_LIMIT_H).fillna(0)
    g['gust_mph'] = g['gust_mph'].fillna(g['wind_mph'])
    g['gust_excess'] = (g['gust_mph'] - g['wind_mph']).clip(lower=0)
    # Backward-looking 3-hour summaries
    g['precip_3h'] = g['precip_in'].rolling(3, min_periods=1).sum()
    g['vis_min_3h'] = g['visibility_mi'].rolling(3, min_periods=1).min()
    g['ceil_min_3h'] = g['ceiling_ft'].rolling(3, min_periods=1).min()
    g['wind_max_3h'] = g['wind_mph'].rolling(3, min_periods=1).max()
    g['thunder_3h'] = g['has_thunder'].rolling(3, min_periods=1).max()
    g['slp_tend_3h'] = g['slp'] - g['slp'].shift(3)
    grids.append(g.reset_index())

df_final = pd.concat(grids, ignore_index=True)
del grids
df_final['precip_in'] = df_final['precip_in'].fillna(0)
print(f'Full grid: {len(df_final):,} rows; hours without reports: {df_final["wx_missing"].mean():.2%}')

Full grid: 6,135,358 rows; hours without reports: 0.21%


### FAA flight category
The category is the worse of the ceiling and visibility criteria:

| Category | Ceiling | or Visibility |
|---|---|---|
| LIFR | < 500 ft | < 1 mi |
| IFR | 500 – < 1,000 ft | 1 – < 3 mi |
| MVFR | 1,000 – 3,000 ft | 3 – 5 mi |
| VFR | > 3,000 ft | > 5 mi |

`flight_cat_ord` encodes VFR = 0 … LIFR = 3 (−1 = missing).

In [12]:
c, v = df_final['ceiling_ft'], df_final['visibility_mi']
cat = np.select(
    [(c < 500) | (v < 1), (c < 1000) | (v < 3), (c <= 3000) | (v <= 5), c.notna() & v.notna()],
    ['LIFR', 'IFR', 'MVFR', 'VFR'], default='UNK')
df_final['flight_cat'] = pd.Categorical(cat, categories=['VFR', 'MVFR', 'IFR', 'LIFR', 'UNK'])
df_final['flight_cat_ord'] = (df_final['flight_cat'].cat.codes
                              .where(df_final['flight_cat'] != 'UNK', -1).astype('int8'))
print((df_final['flight_cat'].value_counts(normalize=True) * 100).round(2))

flight_cat
VFR     79.16
MVFR    10.72
IFR      5.15
LIFR     3.51
UNK      1.47
Name: proportion, dtype: float64


### Save and summary statistics
Writes `data/processed/weather_hourly_utc.parquet` and prints the hourly frequency of each weather flag (all station-hours, 2018–2022).

In [13]:
float_cols = df_final.select_dtypes('float64').columns
df_final[float_cols] = df_final[float_cols].astype('float32')
df_final.to_parquet(WEATHER_CLEAN_PATH, index=False)
print(f'saved {len(df_final):,} rows x {df_final.shape[1]} columns -> {WEATHER_CLEAN_PATH}')

print('\nWeather flag frequency (% of hours):')
print((df_final[FLAG_COLS].mean() * 100).round(2).to_string())
print('\nAdverse-weather hours by state:')
print(df_final.groupby('state')[['has_rain', 'has_snow', 'has_fog', 'has_thunder', 'has_freezing']].sum())

saved 6,135,358 rows x 37 columns -> c:\Users\Admin\Downloads\on-going assement 1\source_code - Copy\data\processed\weather_hourly_utc.parquet

Weather flag frequency (% of hours):
has_rain          6.01
has_snow          1.38
has_fog           1.35
has_mist          6.77
has_thunder       0.79
has_thunder_vc    0.70
has_freezing      0.24
is_heavy          0.67

Adverse-weather hours by state:
       has_rain  has_snow  has_fog  has_thunder  has_freezing
state                                                        
CA      32500.0    1682.0  16387.0        390.0         151.0
CO      13080.0   19323.0   3844.0       1847.0        2768.0
FL      51874.0      30.0  10225.0      17279.0           6.0
GA      21675.0      69.0   6051.0       4056.0          18.0
IL      34730.0   13914.0   5017.0       4202.0        2366.0
NC      32054.0     962.0   5718.0       3092.0         336.0
NY      64060.0   36358.0  11233.0       2888.0        3402.0
TX      55553.0    2244.0  11071.0      1156